In [6]:
"""
@author Thomas Torres
CSC 461/ DSP 461, LocalMAP Silhouette Evaluations, SVM Evaluations,
and recreations of Visualizations

To begin evaluations, start by adding the zipped data file to the root directory.
Unzip it.
Specify its name here, for our evaluations we used 'data.zip'.
Check the GitHub repo for this file, and unzip it here.
"""

ZIPPED_FILE_NAME : str = "data.zip"
#unzip file
!unzip "data.zip" -d "data"
DATA_FILE : str = "data/data/"

#all imports used in the project, unfortunately we could not get LargeVIS to work
#it has a fairly difficult installation process without much documentation.
import numpy as np
from sklearn import datasets
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
from sklearn.metrics import silhouette_score
from scipy.stats import norm
import time
import os

#install all relevant libraries used in the paper
libraries=['phate','opentsne',
'umap-learn',
'hnne',
'contrastive-ne',
'trimap',
'tqdm',
'pacmap']
for lib in libraries:
  !pip install {lib}

Archive:  data.zip
  inflating: data/data/20NG.npy      
  inflating: data/data/20NG_labels.npy  
  inflating: data/data/coil_20.npy   
  inflating: data/data/coil_20_labels.npy  
  inflating: data/data/fmnist_images.npy  
  inflating: data/data/fmnist_labels.npy  
  inflating: data/data/mnist_images.npy  
  inflating: data/data/mnist_labels.npy  
  inflating: data/data/Preprocessing.ipynb  
  inflating: data/data/USPS.npy      
  inflating: data/data/USPS_labels.npy  


In [ ]:
"""Function that is slightly modified code from the LocalMAP repository for evaluating
different models on data. Produces and saves an embedding to a specific directory.
@param model_name - name of the model we want to evaluate data on
@param data_set_name - what to save the embedded data as
@i - index of the saved data
@X - the data we want to embed
@return Returns the transformed data
"""
def evaluate_model(model_name : str, data_set_name : str, i : int, X,y ):
    if model_name == 'PaCMAP':
        import pacmap
        model = pacmap.PaCMAP()
        start_time = time.time()
        X_trans = model.fit_transform(X)
        total_time = time.time()-start_time
    elif model_name == "LocalMAP":
        from pacmap import LocalMAP
        model = LocalMAP()
        start_time = time.time()
        X_trans = model.fit_transform(X)
        total_time = time.time()-start_time
    elif model_name == "TSNE":
        # import openTSNE
        # model = openTSNE.TSNE(n_jobs=-1)
        from sklearn.manifold import TSNE
        model = TSNE()
        start_time = time.time()
        X_trans = model.fit_transform(X)
        total_time = time.time()-start_time
    elif model_name == "UMAP":
        import umap
        model = umap.UMAP()
        start_time = time.time()
        X_trans = model.fit_transform(X)
        total_time = time.time()-start_time
    elif model_name == "HNNE":
        import hnne
        model = hnne.HNNE()
        start_time = time.time()
        X_trans = model.fit_transform(X)
        total_time = time.time()-start_time
    elif model_name == "InfoNCE":
        import cne
        model = cne.CNE()
        start_time = time.time()
        X_trans = model.fit_transform(X.astype(float))
        total_time = time.time()-start_time
    elif model_name == "NegTSNE":
        import cne
        model = cne.CNE(loss_mode="neg")
        start_time = time.time()
        X_trans = model.fit_transform(X.astype(float))
        total_time = time.time()-start_time
    elif model_name == "NCVi":
        import cne
        model = cne.CNE(loss_mode="nce",optimizer="adam")
        start_time = time.time()
        X_trans = model.fit_transform(X.astype(float))
        total_time = time.time()-start_time
    elif model_name == "TriMAP":
        import trimap
        model = trimap.TRIMAP()
        start_time = time.time()
        X_trans = model.fit_transform(X)
        total_time = time.time()-start_time
    elif model_name == "PHATE":
        import phate
        model = phate.PHATE(n_jobs=-1)
        start_time = time.time()
        X_trans = model.fit_transform(X)
        total_time = time.time()-start_time
    elif model_name == "PCA":
        from sklearn.decomposition import PCA
        model = PCA(n_components=2)
        start_time = time.time()
        X_trans = model.fit_transform(X)
        total_time = time.time()-start_time

    # save the embedding into the embedding folder
    #was supposed to this originally but i decided to save it in the function that
    #calls this function
    return (X_trans,total_time)

"""Evaluation Function for a specific dataset"""
def eval_data(data_name : str,data_set,labels):

    models=["PaCMAP","LocalMAP","TSNE","UMAP","HNNE","InfoNCE","NegTSNE","NCVi","TriMAP","PHATE","PCA"]
    fig, axes = plt.subplots(3, 4, figsize=(10, 10))
    axes = axes.flatten()
    for i in range(len(models)):
      total_time = 0
      print(f"Evaluating: {models[i]}")
      for j in range(10):
            model_name = models[i]
            SAVE_DIR : str = f"embeddings/{model_name}/{data_name}_embedding_{j+1}.npy"
            #10 diff trials with each model
            X_trans,time_increment=evaluate_model(model_name, data_name, i,data_set,labels)
            total_time+=time_increment
            #save embedding
            np.save(SAVE_DIR,X_trans)

            #if we are on last iteration of j then render
            if(j+1==10):
              ax = axes[i]
              ax.scatter(X_trans[:, 0], X_trans[:, 1], cmap="Spectral", c=y, s=1)
              ax.set_title(f"{model_name}")
              ax.set_xlabel("X")
              ax.set_ylabel("Y")
          #I kept running into errors with HNNE for low quantities of basis vectors
          #when considering transformed data, so this is just to handle it
      total_time/=10
      print(f"Mean time for {models[i]}: {total_time}")
      #save embedding

    fig.suptitle(data_name, fontsize=16, fontweight='bold')
    #show layout
    plt.tight_layout()
    plt.legend()
    plt.show()

    #save resulting figure
    plt.savefig(f"{data_name}.png")


#set up the embeddings folder and subfolders
!mkdir embeddings
models=["PaCMAP","LocalMAP","TSNE","UMAP","HNNE","InfoNCE","NegTSNE","NCVi","TriMAP","PHATE","PCA"]
for model in models:
  !mkdir embeddings/{model}
total_times = {}
#finally evaluate eigen faces on it
file_name_roots = ["20NG","USPS","coil_20","fmnist","mnist"]
for root in file_name_roots:
    data_file_name : str = f"{DATA_FILE}{root}.npy"
    label_file_name : str = f"{DATA_FILE}{root}_labels.npy"

    #load data
    X=np.load(data_file_name,allow_pickle=True)
    y=np.load(label_file_name,allow_pickle=True)

    #evaluate and produce image
    new_time=eval_data(root,X,y)

    #create total time matrix
    if(root not in total_times):
      total_times[root]=[new_time]
    else:
      total_times[root].append(new_time)




mkdir: cannot create directory ‘embeddings’: File exists
mkdir: cannot create directory ‘embeddings/PaCMAP’: File exists
mkdir: cannot create directory ‘embeddings/LocalMAP’: File exists
mkdir: cannot create directory ‘embeddings/TSNE’: File exists
mkdir: cannot create directory ‘embeddings/UMAP’: File exists
mkdir: cannot create directory ‘embeddings/HNNE’: File exists
mkdir: cannot create directory ‘embeddings/InfoNCE’: File exists
mkdir: cannot create directory ‘embeddings/NegTSNE’: File exists
mkdir: cannot create directory ‘embeddings/NCVi’: File exists
mkdir: cannot create directory ‘embeddings/TriMAP’: File exists
mkdir: cannot create directory ‘embeddings/PHATE’: File exists
mkdir: cannot create directory ‘embeddings/PCA’: File exists
Evaluating: PaCMAP
Mean time for PaCMAP: 14.350545811653138
Evaluating: LocalMAP
Mean time for LocalMAP: 19.58687973022461
Evaluating: TSNE
Mean time for TSNE: 284.0543601989746
Evaluating: UMAP
Mean time for UMAP: 16.90439054965973
Evaluating: HN

Finished epoch 0/200, loss 12223.659
Finished epoch 40/200, loss 12854.857
Finished epoch 80/200, loss 7567.150
Finished epoch 120/200, loss 6923.327
Finished epoch 160/200, loss 6734.624


Computing approximate kNN graph with annoy


Finished epoch 0/200, loss 12223.659
Finished epoch 40/200, loss 12854.857
Finished epoch 80/200, loss 7567.150
Finished epoch 120/200, loss 6923.327
Finished epoch 160/200, loss 6734.624


Computing approximate kNN graph with annoy


Finished epoch 0/200, loss 12223.659
Finished epoch 40/200, loss 12854.857
Finished epoch 80/200, loss 7567.150
Finished epoch 120/200, loss 6923.327
Finished epoch 160/200, loss 6734.624


Computing approximate kNN graph with annoy


Finished epoch 0/200, loss 12223.659
Finished epoch 40/200, loss 12854.857
Finished epoch 80/200, loss 7567.150
Finished epoch 120/200, loss 6923.327
Finished epoch 160/200, loss 6734.624


Computing approximate kNN graph with annoy


Finished epoch 0/200, loss 12223.659
Finished epoch 40/200, loss 12854.857


In [ ]:
"""This step is meant to perform all of the evaluations."""

#in collab, create a folder for all of the saved embeddings and their
!mkdir embeddings
models=["PaCMAP","LocalMAP","TSNE","UMAP","HNNE","InfoNCE","NegTSNE","NCVi","TriMAP","PHATE","PCA"]
for model in models:
  !mkdir embeddings/{model}

#for each dataset, evaluate every model on them